In [4]:
# !pip install llama-index==0.12.9 llama-index-llms-ollama==0.5.0 llama-index-embeddings-huggingface==0.4.0 llama-index-llms-groq==0.3.1

In [5]:
import os

In [6]:
import nest_asyncio

nest_asyncio.apply()

In [7]:
from llama_index.core import SimpleDirectoryReader

In [8]:
# load teh document
documents = SimpleDirectoryReader(input_files=["attention_is_all_you_need.pdf"]).load_data()

In [10]:
print(type(documents))

<class 'list'>


In [11]:
len(documents)

15

In [15]:
#documents[0]

In [16]:
from llama_index.core.node_parser import SentenceSplitter

In [17]:
splitter = SentenceSplitter(chunk_size=2000,
                           chunk_overlap=20)

In [18]:
nodes = splitter.get_nodes_from_documents(documents)

In [19]:
print(type(nodes))

<class 'list'>


In [23]:
from llama_index.core import Settings
#from llama_index.llms.ollama import Ollama
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [24]:
#Settings.llm = Ollama(model="gemma:2b")

In [25]:
os.environ["GROQ_API_KEY"] = "your_groq_api_key"

In [26]:
Settings.llm = Groq(model="llama-3.1-8b-instant")

In [27]:
Settings.embed_model = HuggingFaceEmbedding()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\siddh\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\siddh\AppData\Local\llama_index\models--BAAI--bge-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
from llama_index.core import SummaryIndex, VectorStoreIndex

In [29]:
summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

In [31]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True
)

vector_query_engine = vector_index.as_query_engine()

In [32]:
from llama_index.core.tools import QueryEngineTool

In [33]:
summary_tool = QueryEngineTool.from_defaults(
    query_engine = summary_query_engine,
    description = (
        "Useful for summarization related to the  given context"
    )
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine = vector_query_engine,
    description = (
        "Useful for retrieving specific context from the given context based on the given question"
    )
)

In [35]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

In [37]:
query_engine = RouterQueryEngine(
    selector = LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool
    ],
    verbose=True
)

In [38]:
response = query_engine.query("Summarize the given document")
print(response)

Selecting query engine 0: The question is asking for a summary of the given document, which matches the description in choice 1..
A new approach to sequence transduction models has been proposed, which relies on self-attention mechanisms to draw global dependencies between input and output sequences. This model, called the Transformer, has achieved state-of-the-art results in various natural language processing tasks, including machine translation and constituency parsing. It allows for significantly more parallelization and can reach new levels of translation quality after a short training period. The Transformer architecture consists of an encoder and a decoder, both composed of multiple identical layers, and uses residual connections and layer normalization to facilitate its sub-layers. The model uses multi-head attention in three different ways and positional encoding to inject information about the relative or absolute position of tokens in the sequence. Experimental results show 

In [39]:
response = query_engine.query("What is the model architecture mentioned in teh paper?")
print(response)

Selecting query engine 1: The question is asking for a specific piece of information (the model architecture) from the given context (the paper), which matches option 2..
The Transformer model architecture follows a stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder and decoder are composed of a stack of identical layers, each containing two sub-layers. The first sub-layer is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. Residual connections and layer normalization are employed around each of the two sub-layers in the encoder and decoder.
